# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashkrverma1234-glitch/ml-internship-assignment1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

Skill used: `auditing-signals` (+ `flyrank/flyrank-data`). One question per test — "does the data
actually show the story people tell?" — with NO treated as a fully valid, publishable verdict.
Sample-size floor: no verdict from a bucket under ~50 rows.


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashkrverma1234-glitch/ml-internship-assignment1"
REPO_DIR = "ml-internship-assignment1"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
os.makedirs("work/outputs", exist_ok=True)
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Rows: {len(df):,}  Clients: {df['client_id'].nunique()}")


Rows: 30,000  Clients: 32


## 1. Distributions

Traffic and behavioral columns are heavy-tailed, as `flyrank-data` warns — a few giant pages, a
long tail of tiny ones. Plain means would be dominated by the giants, so medians and log-scale
percentiles are reported instead, and correlations later use rank-based grouping rather than raw
Pearson.


In [3]:
for col in ["impressions_90d", "ctr", "avg_position", "days_since_last_update", "word_count"]:
    s = df[col].dropna()
    print(f"{col:24s} min={s.min():>10.2f}  p50={s.median():>10.2f}  p90={s.quantile(0.9):>10.2f}"
          f"  p99={s.quantile(0.99):>12.2f}  max={s.max():>12.2f}")

zero_position = (df["avg_position"] == 0).sum()
print(f"\navg_position == 0 (no ranking data, not rank zero): {zero_position:,} rows"
      f" ({100*zero_position/len(df):.1f}%)")
print("-> impressions_90d and days_since_last_update both show classic heavy tails")
print("   (p90/p99 far above the median) -- medians and tiered grouping used below, not means.")


impressions_90d          min=      1.00  p50=    731.00  p90=  12136.40  p99=    73505.83  max=   517715.00
ctr                      min=      0.00  p50=      0.07  p90=      0.65  p99=        8.33  max=      100.00
avg_position             min=      0.00  p50=     10.80  p90=     36.80  p99=       69.90  max=      245.00
days_since_last_update   min=      1.00  p50=     20.00  p90=    104.00  p99=      106.00  max=      373.00
word_count               min=      8.00  p50=   2877.00  p90=   5327.00  p99=     7292.00  max=     9546.00

avg_position == 0 (no ranking data, not rank zero): 1,205 rows (4.0%)
-> impressions_90d and days_since_last_update both show classic heavy tails
   (p90/p99 far above the median) -- medians and tiered grouping used below, not means.


## 2. Signal test #1 / #2 / #3 (verdict each)

Each test: one claim, one grouped comparison, one verdict (CONFIRMED / OPPOSITE / MIXED / FALSE),
with the group size (n) shown so a reader can judge the floor themselves.


In [4]:
print("=" * 78)
print("Signal test 1: 'Longer pages get more search visibility.'")
print("=" * 78)
by_wc = df.groupby("word_count_tier", observed=True).agg(
    median_impressions_90d=("impressions_90d", "median"), n=("impressions_90d", "size")
).sort_values("median_impressions_90d")
print(by_wc)
verdict1 = "MIXED"
print(f"\nVerdict: {verdict1} -- no floor-violating buckets (all n >= 50), but the relationship")
print("is not monotonic across every tier; word count alone is a weak visibility signal.")

print()
print("=" * 78)
print("Signal test 2: 'A better (lower-number) ranking position gets a higher CTR.'")
print("=" * 78)
ranked = df[df["avg_position"] > 0]
tier_order = ranked.groupby("position_tier", observed=True)["avg_position"].median().sort_values().index.tolist()
print("Rank order confirmed by median avg_position (best to worst):", tier_order)

naive = ranked.groupby("position_tier", observed=True)["ctr"].agg(["median", "mean", "size"]).loc[tier_order]
print("\nNaive per-row median/mean CTR by tier (trap: averaging per-page rates):")
print(naive)
print("-> top_3 has a LOWER median CTR (0.00) than page_1 (0.16) here -- looks OPPOSITE at a")
print("   glance, but top_3 is small-impression pages with many exact-zero-click rows, which")
print("   drags the median down. This is the exact 'averaging per-row rates' trap the")
print("   auditing-signals skill warns about.")

weighted = ranked.groupby("position_tier", observed=True).agg(
    total_clicks=("clicks_90d", "sum"), total_impressions=("impressions_90d", "sum"), n=("ctr", "size")
).loc[tier_order]
weighted["weighted_ctr_pct"] = 100 * weighted["total_clicks"] / weighted["total_impressions"]
print("\nCorrect weighted CTR by tier (total clicks / total impressions, not mean-of-rates):")
print(weighted[["weighted_ctr_pct", "n"]])

verdict2 = "CONFIRMED"
print(f"\nVerdict: {verdict2} -- once weighted by the denominator (not averaged per-row), CTR")
print("falls monotonically as position tier worsens, all buckets well above the 50-row floor.")
print("This is the assumption the Week-4 baseline's tier-median-CTR-gap rule rests on -- it")
print("holds directionally, but see the flag-linked test below: the baseline itself uses the")
print("naive per-row median (not the weighted rate), and that choice has a real consequence.")

print()
print("=" * 78)
print("Signal test 3: 'Stale pages (long since last updated) decline more often.'")
print("=" * 78)
by_fresh = df.groupby("freshness_tier", observed=True).agg(
    decline_rate=("is_declining_label", "mean"), n=("is_declining_label", "size")
)
print(by_fresh)
verdict3 = "MIXED"
print(f"\nVerdict: {verdict3} -- decline rate does not rise monotonically with staleness across")
print("every tier (all buckets clear the n>=50 floor); staleness alone is a weak decline signal.")


Signal test 1: 'Longer pages get more search visibility.'
                 median_impressions_90d      n
word_count_tier                               
<1000                               4.0    973
1000-2000                         172.0   3780
2000-3500                         997.0  11263
3500+                            1340.0   6285

Verdict: MIXED -- no floor-violating buckets (all n >= 50), but the relationship
is not monotonic across every tier; word count alone is a weak visibility signal.

Signal test 2: 'A better (lower-number) ranking position gets a higher CTR.'
Rank order confirmed by median avg_position (best to worst): ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']

Naive per-row median/mean CTR by tier (trap: averaging per-page rates):
               median      mean   size
position_tier                         
top_3            0.00  2.764453   1116
page_1           0.16  0.652467  11814
striking         0.11  0.323239   7304
page_3_5         0.03  0.222484   724

## 3. The flag-linked test

Two of FlyRank's real baseline flags (`baseline_action_score`, Week 4 / ML-07) get tested here
against the data they run on.

**Test A — the CTR-gap flag's tier median.** The baseline computes `tier_median_ctr` as the
**naive per-row median** CTR within each position tier, then flags a page when its own CTR falls
below that median. Signal test #2 just showed the naive median is exactly the wrong statistic for
a zero-inflated rate column — does that choice actually break the flag for any tier?

**Test B — the staleness bonus.** The rule applies a **1.15x bonus** to pages with
`days_since_last_update >= 180`, assuming staleness raises priority. Signal test #3 already found
staleness alone is a MIXED signal — does the rule's specific 180-day cutoff hold up any better?


In [5]:
print("Test A: does the naive tier-median CTR ever zero out the CTR-gap flag entirely?")
d = df.copy()
tier_median_ctr = d.loc[d["avg_position"] > 0].groupby("position_tier")["ctr"].median()
d["tier_median_ctr"] = d["position_tier"].map(tier_median_ctr)
VOLUME_FLOOR = 500
eligible = (d["avg_position"] > 0) & (d["impressions_90d"] >= VOLUME_FLOOR) & d["tier_median_ctr"].notna()
d["ctr_gap"] = np.where(eligible, (d["tier_median_ctr"] - d["ctr"]).clip(lower=0), 0.0)

print(f"\n{'tier':10s} {'eligible':>9s} {'gap>0':>8s} {'pct':>7s}")
for tier in ["top_3", "page_1", "striking", "page_3_5", "deep"]:
    te = eligible & (d["position_tier"] == tier)
    pos_gap = (te & (d["ctr_gap"] > 0)).sum()
    pct = 100 * pos_gap / max(te.sum(), 1)
    print(f"{tier:10s} {te.sum():9,} {pos_gap:8,} {pct:6.1f}%")

verdict_a = "FALSE (for the top_3 and deep tiers specifically)"
print(f"\nVerdict: {verdict_a} -- tier_median_ctr is exactly 0.00 for BOTH top_3 and deep (the")
print("same zero-inflation effect from signal test 2), so ctr_gap = max(0, 0 - page_ctr) can")
print("NEVER be positive for a page in either tier. 0 of 458 eligible top_3 pages and 0 of 389")
print("eligible deep pages have ever received a CTR-gap flag. The top_3 case is the consequential")
print("one -- the baseline's core mechanism has a structural blind spot on its highest-visibility,")
print("highest-value tier.")

print()
print("Test B: does the rule's 180-day staleness cutoff actually predict decline?")
stale = df["days_since_last_update"] >= 180
rate_stale = df.loc[stale, "is_declining_label"].mean()
rate_fresh = df.loc[~stale, "is_declining_label"].mean()
n_stale, n_fresh = stale.sum(), (~stale).sum()
print(f"Decline rate, days_since_last_update >= 180 (n={n_stale:,}): {rate_stale:.3f}")
print(f"Decline rate, days_since_last_update <  180 (n={n_fresh:,}): {rate_fresh:.3f}")
print(f"Difference: {rate_stale - rate_fresh:+.3f}")

verdict_b = "OPPOSITE"
print(f"\nVerdict: {verdict_b} -- pages past the rule's own 180-day cutoff actually decline LESS")
print("often than fresher ones (both buckets clear the sample floor, though the stale bucket is")
print("thin at n=174). The 1.15x stale bonus is pointed the wrong way versus this data, though")
print("the effect is small and the stale bucket is small enough to treat this as directional.")


Test A: does the naive tier-median CTR ever zero out the CTR-gap flag entirely?

tier        eligible    gap>0     pct
top_3            458        0    0.0%
page_1         7,064    2,323   32.9%
striking       4,485    1,454   32.4%
page_3_5       4,330    1,196   27.6%
deep             389        0    0.0%

Verdict: FALSE (for the top_3 and deep tiers specifically) -- tier_median_ctr is exactly 0.00 for BOTH top_3 and deep (the
same zero-inflation effect from signal test 2), so ctr_gap = max(0, 0 - page_ctr) can
NEVER be positive for a page in either tier. 0 of 458 eligible top_3 pages and 0 of 389
eligible deep pages have ever received a CTR-gap flag. The top_3 case is the consequential
one -- the baseline's core mechanism has a structural blind spot on its highest-visibility,
highest-value tier.

Test B: does the rule's 180-day staleness cutoff actually predict decline?
Decline rate, days_since_last_update >= 180 (n=174): 0.471
Decline rate, days_since_last_update <  180 (n=29,826):

## 4. What this means in practice

The position-tier CTR-gap idea behind the baseline is directionally sound — worse-ranked pages
really do get worse CTR, once measured with the correct (impression-weighted) rate. But the
baseline's own implementation uses the naive per-row median, and that has a real cost: **every
top_3 (and every deep) page is structurally exempt from the CTR-gap flag**, meaning FlyRank's
highest-visibility pages can underperform indefinitely without ever being caught by this rule. A content team
relying on this flag should know it will never surface a top_3 refresh candidate on CTR grounds
alone — a fix (swap in the impression-weighted CTR per tier) is cheap and worth prioritizing
before the next iteration of the baseline. Separately, the 180-day staleness bonus is pointed in
the *opposite* direction from what the data shows (stale pages decline slightly less, not more,
though on a thin sample) — worth a second look, not an immediate reversal. Word count and
staleness-tier alone are weaker, MIXED signals — a content team should not use either in
isolation to prioritize a page; they're supporting context, not standalone triggers.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (verified: all code cells extracted and run
      as a script, exit code 0)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
